# 03 数据挖掘与模式发现

本 Notebook 基于 EDA 结论继续挖掘可用于推荐优化的模式，所有统计均以 `2020-09-16` 之前的交易作为历史，最后 7 天 `2020-09-16` 至 `2020-09-22` 仅作为测试标签，避免时间泄漏。

**挖掘内容：**

- 用户-商品复购频率、最近购买时间与指数时间衰减得分；
- 同一天购物篮关联规则与同一周多天购买关联规则；
- 近 60 天隐式反馈矩阵的随机化 SVD 商品相似性；
- 用户类目、价格带、颜色偏好与分群热门商品；
- 最后 7 天测试集上的 Recall@12、HitRate@12、MAP@12 与候选覆盖率。

输出保存在 `optimization/output/mining/`，由 `04_final_candidate_feature_model.ipynb` 的多路召回与精排特征工程复用。

In [ ]:
import json
import os
import time
import warnings
from collections import defaultdict
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-hm-recsys')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from scipy.sparse.linalg import svds

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 160
plt.rcParams['savefig.bbox'] = 'tight'
np.random.seed(2026)


def find_clean_dir():
    candidates = [
        Path.cwd() / 'output' / 'clean',
        Path.cwd() / 'optimization' / 'output' / 'clean',
        Path.cwd().parent / 'output' / 'clean',
        Path.cwd().parent / 'optimization' / 'output' / 'clean',
        Path('/kaggle/input/hm-recsys-clean'),
        Path('/kaggle/input/hm_recsys_clean'),
    ]
    required = {'transactions_clean.parquet', 'customers_clean.parquet', 'articles_clean.parquet', 'submission_customers.parquet'}
    for candidate in candidates:
        if candidate.is_dir() and required.issubset({path.name for path in candidate.iterdir()}):
            return candidate.resolve()
    return None


def find_raw_dir():
    candidates = [
        Path.cwd() / 'data',
        Path.cwd().parent / 'data',
        Path.cwd().parents[1] / 'data',
        Path('/kaggle/input/h-and-m-personalized-fashion-recommendations'),
    ]
    required = {'transactions_train.csv', 'customers.csv', 'articles.csv', 'sample_submission.csv'}
    for candidate in candidates:
        if candidate.is_dir() and required.issubset({path.name for path in candidate.iterdir()}):
            return candidate.resolve()
    raise FileNotFoundError(f'未找到清洗产物或原始数据: {sorted(required)}')


def read_raw_fallback(raw_dir):
    print('未找到清洗产物，回退读取原始数据并执行最小化规范化...')
    with (raw_dir / 'articles.csv').open('rb') as article_file:
        compression = 'zip' if article_file.read(4) == b'PK\x03\x04' else None
    tx = pd.read_csv(raw_dir / 'transactions_train.csv', dtype={'customer_id': 'string', 'article_id': 'string'})
    tx['t_dat'] = pd.to_datetime(tx['t_dat'], format='%Y-%m-%d')
    tx['customer_key'] = np.fromiter((int(value[-16:], 16) for value in tx['customer_id']), dtype=np.uint64, count=len(tx))
    tx['article_num'] = tx['article_id'].str.zfill(10).astype('int32')
    tx['price'] = pd.to_numeric(tx['price'], errors='raise').astype('float32')
    tx['sales_channel_id'] = pd.to_numeric(tx['sales_channel_id'], errors='raise').astype('int8')
    tx = tx[['t_dat', 'customer_key', 'article_num', 'price', 'sales_channel_id']]
    customers = pd.read_csv(raw_dir / 'customers.csv', dtype={'customer_id': 'string'})
    customers['customer_key'] = np.fromiter((int(value[-16:], 16) for value in customers['customer_id']), dtype=np.uint64, count=len(customers))
    customers = customers.rename(columns={'FN': 'fn_flag', 'Active': 'active_flag'})
    articles = pd.read_csv(raw_dir / 'articles.csv', compression=compression, dtype={'article_id': 'string', 'product_code': 'int32'})
    articles['article_num'] = articles['article_id'].str.zfill(10).astype('int32')
    submission = pd.read_csv(raw_dir / 'sample_submission.csv', usecols=['customer_id'], dtype={'customer_id': 'string'})
    submission['customer_key'] = np.fromiter((int(value[-16:], 16) for value in submission['customer_id']), dtype=np.uint64, count=len(submission))
    return tx, customers, articles[['article_num', 'product_code', 'prod_name', 'product_type_name', 'product_group_name', 'index_group_name', 'department_name', 'colour_group_name']], submission


CLEAN_DIR = find_clean_dir()
RAW_DIR = None if CLEAN_DIR else find_raw_dir()
if CLEAN_DIR:
    transactions = pd.read_parquet(CLEAN_DIR / 'transactions_clean.parquet', columns=['t_dat', 'customer_key', 'article_id', 'price', 'sales_channel_id'])
    customers = pd.read_parquet(CLEAN_DIR / 'customers_clean.parquet')
    articles = pd.read_parquet(CLEAN_DIR / 'articles_clean.parquet')
    submission = pd.read_parquet(CLEAN_DIR / 'submission_customers.parquet')
    transactions['article_num'] = transactions['article_id'].astype('int32')
    transactions = transactions.drop(columns='article_id')
else:
    transactions, customers, articles, submission = read_raw_fallback(RAW_DIR)
   
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
articles['article_num'] = articles['article_id'].astype('int32') if 'article_id' in articles else articles['article_num'].astype('int32')
article_meta = articles[['article_num', 'product_code', 'prod_name', 'product_type_name', 'product_group_name', 'index_group_name', 'department_name', 'colour_group_name']].copy()

LOCAL_PROJECT_DIR = next((base / 'optimization' for base in (Path.cwd(), *Path.cwd().parents) if (base / 'optimization' / 'code').is_dir()), Path.cwd())
MINING_DIR = Path('/kaggle/working/hm_recsys_mining') if Path('/kaggle/working').exists() else LOCAL_PROJECT_DIR / 'output' / 'mining'
FIGURE_DIR = MINING_DIR / 'figures'
TABLE_DIR = MINING_DIR / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_END = transactions['t_dat'].max()
TARGET_START = TARGET_END - pd.Timedelta(days=6)
history = transactions[transactions['t_dat'] < TARGET_START].copy()
target = transactions[transactions['t_dat'] >= TARGET_START].copy()
history['days_to_target'] = (TARGET_START - history['t_dat']).dt.days.astype('int16')
target_users = np.sort(target['customer_key'].unique())
target_user_set = set(target_users)

print('CLEAN_DIR:', CLEAN_DIR)
print('transactions:', transactions.shape)
print('history:', history.shape, history['t_dat'].min().date(), history['t_dat'].max().date())
print('target:', target.shape, TARGET_START.date(), TARGET_END.date())
print('target users:', len(target_users))
display(article_meta.head())

## 1. 时间衰减复购模式挖掘

EDA 显示同商品复购的中位间隔为 8 天，但用户整体活跃间隔为 22 天，说明复购信号集中在较短的局部窗口。这里为每个用户-商品统计购买次数、首次/最近购买时间、价格偏好与季节性购买，并比较多个指数衰减系数 $\alpha$：

$$score(u,i)=\log(1+count(u,i))\times \alpha^{\Delta t}$$

其中 $\Delta t$ 是目标日开始前距最近一次购买的天数。

In [ ]:
def top_k_by_user(frame, score_column, customer_column='customer_key', article_column='article_num', top_n=12):
    ordered = frame.sort_values([customer_column, score_column, article_column], ascending=[True, False, True], kind='stable')
    ordered['_user_rank'] = ordered.groupby(customer_column, sort=False).cumcount() + 1
    return ordered[ordered['_user_rank'] <= top_n].drop(columns='_user_rank')


def evaluate_candidates(name, candidates, truth, target_user_count, k=12, history_pairs=None, show_output=True):
    required = {'customer_key', 'article_num', 'score'}
    if not required.issubset(candidates.columns):
        raise ValueError(f'{name} 缺少字段: {required - set(candidates.columns)}')
    ranked = candidates[['customer_key', 'article_num', 'score']].copy()
    ranked = ranked.sort_values(['customer_key', 'score', 'article_num'], ascending=[True, False, True], kind='stable')
    ranked = ranked.drop_duplicates(['customer_key', 'article_num'], keep='first')
    ranked['_rank'] = ranked.groupby('customer_key', sort=False).cumcount() + 1
    ranked = ranked[ranked['_rank'] <= k]
    evaluated = ranked.merge(truth[['customer_key', 'article_num', 'label']], on=['customer_key', 'article_num'], how='left')
    evaluated['label'] = evaluated['label'].fillna(0).astype('int8')
    evaluated['cumulative_hits'] = evaluated.groupby('customer_key', sort=False)['label'].cumsum()
    evaluated['precision_at_hit'] = evaluated['label'] * evaluated['cumulative_hits'] / evaluated['_rank']
    user_hit = evaluated.groupby('customer_key').agg(hit_sum=('label', 'sum'), max_rank=('_rank', 'max'), ap_sum=('precision_at_hit', 'sum')).reset_index()
    user_truth = truth.groupby('customer_key')['label'].sum().rename('truth_count')
    user_hit = user_hit.merge(user_truth, on='customer_key', how='right').fillna({'hit_sum': 0, 'max_rank': 0, 'ap_sum': 0})
    user_hit['recall'] = user_hit['hit_sum'] / user_hit['truth_count'].clip(lower=1)
    user_hit['average_precision'] = user_hit['ap_sum'] / user_hit['truth_count'].clip(upper=k).clip(lower=1)
    novel_share = None
    if history_pairs is not None:
        novelty = ranked.merge(history_pairs[['customer_key', 'article_num', 'in_history']], on=['customer_key', 'article_num'], how='left')
        novelty['in_history'] = novelty['in_history'].fillna(0).astype('int8')
        novel_share = float(1 - novelty['in_history'].mean())
    result = {
        'channel': name,
        'map@12': float(user_hit['average_precision'].mean()),
        'recall@12': float(user_hit['recall'].mean()),
        'hit_rate@12': float((user_hit['hit_sum'] > 0).mean()),
        'covered_target_users': int(ranked['customer_key'].nunique()),
        'target_user_coverage': float(ranked['customer_key'].nunique() / target_user_count),
        'candidate_rows': int(len(ranked)),
        'novel_item_share': novel_share,
    }
    if show_output:
        print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


temporal_started = time.time()
user_article_history = history.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
    purchase_count=('t_dat', 'size'),
    first_date=('t_dat', 'min'),
    last_date=('t_dat', 'max'),
    avg_price=('price', 'mean'),
)
channel_pair_counts = history.groupby(['customer_key', 'article_num', 'sales_channel_id'], observed=True).size().unstack(fill_value=0).reset_index()
channel_pair_counts.columns = ['customer_key', 'article_num', 'channel_1_count', 'channel_2_count']
user_article_history = user_article_history.merge(channel_pair_counts, on=['customer_key', 'article_num'], how='left')
user_article_history[['channel_1_count', 'channel_2_count']] = user_article_history[['channel_1_count', 'channel_2_count']].fillna(0).astype('int32')
user_article_history['days_since_last'] = (TARGET_START - user_article_history['last_date']).dt.days.astype('int16')
user_article_history['days_between_first_last'] = (user_article_history['last_date'] - user_article_history['first_date']).dt.days.astype('int16')
user_article_history['repeat_pair'] = (user_article_history['purchase_count'] > 1).astype('int8')
   
seasonal_rows = history[history['t_dat'].between(TARGET_START - pd.Timedelta(days=386), TARGET_START - pd.Timedelta(days=344))]
seasonal = seasonal_rows.groupby(['customer_key', 'article_num'], as_index=False, observed=True).size().rename(columns={'size': 'seasonal_purchase_count'})
user_article_history = user_article_history.merge(seasonal, on=['customer_key', 'article_num'], how='left')
user_article_history['seasonal_purchase_count'] = user_article_history['seasonal_purchase_count'].fillna(0).astype('int16')
   
history_pair_set = user_article_history[['customer_key', 'article_num']].copy()
history_pair_set['in_history'] = np.int8(1)
target_truth = target[['customer_key', 'article_num']].drop_duplicates()
target_truth['label'] = np.int8(1)
   
alpha_results = []
alpha_candidates = {}
for alpha in [0.985, 0.990, 0.995, 0.998, 0.999]:
    score_name = f'time_decay_alpha_{str(alpha).replace(".", "")}'
    user_article_history[score_name] = np.log1p(user_article_history['purchase_count']) * np.power(alpha, user_article_history['days_since_last'])
    eval_pairs = user_article_history[user_article_history['customer_key'].isin(target_user_set)]
    candidates = top_k_by_user(eval_pairs, score_name, top_n=12)
    candidates = candidates.rename(columns={score_name: 'score'})
    alpha_candidates[alpha] = candidates
    alpha_results.append(evaluate_candidates(f'time_decay_alpha={alpha}', candidates, target_truth, len(target_users), history_pairs=history_pair_set, show_output=False))
   
alpha_summary = pd.DataFrame(alpha_results)
best_alpha = float(alpha_summary.sort_values('recall@12', ascending=False).iloc[0]['channel'].split('=')[1])
best_alpha_score = f'time_decay_alpha_{str(best_alpha).replace(".", "")}'
user_article_history['time_decay_score'] = user_article_history[best_alpha_score]
user_article_history['seasonal_boost_score'] = user_article_history['time_decay_score'] + 0.35 * user_article_history['seasonal_purchase_count']
all_time_decay = top_k_by_user(user_article_history, 'time_decay_score', top_n=12)[['customer_key', 'article_num', 'purchase_count', 'days_since_last', 'time_decay_score', 'seasonal_purchase_count']]
all_time_decay.to_parquet(MINING_DIR / 'time_decay_candidates_top12.parquet', index=False, compression='zstd')
   
seasonal_eval = top_k_by_user(user_article_history[user_article_history['customer_key'].isin(target_user_set)], 'seasonal_boost_score', top_n=12).rename(columns={'seasonal_boost_score': 'score'})
seasonal_result = evaluate_candidates('time_decay_plus_seasonal', seasonal_eval, target_truth, len(target_users), history_pairs=history_pair_set, show_output=False)
alpha_summary = pd.concat([alpha_summary, pd.DataFrame([seasonal_result])], ignore_index=True)
save_table = alpha_summary if False else None
alpha_summary.to_csv(TABLE_DIR / 'time_decay_evaluation.csv', index=False)
   
fig, axis = plt.subplots(figsize=(11, 5))
plot_data = alpha_summary[alpha_summary['channel'].str.contains('alpha=')].copy()
plot_data['alpha'] = plot_data['channel'].str.extract(r'alpha=([0-9.]+)').astype(float)
sns.lineplot(data=plot_data, x='alpha', y='recall@12', marker='o', label='Recall@12', ax=axis)
sns.lineplot(data=plot_data, x='alpha', y='hit_rate@12', marker='o', label='HitRate@12', ax=axis)
axis.set_title('Time-decay repurchase signal by alpha')
axis.set_xlabel('Alpha (larger means slower decay)')
axis.set_ylabel('Score')
axis.xaxis.set_major_formatter(lambda value, _: f'{value:.3f}')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_time_decay_alpha_evaluation.png', bbox_inches='tight')
plt.close(fig)
   
gap_bins = [0, 7, 14, 30, 60, 90, 180, 365, 10_000]
gap_labels = ['0-7d', '8-14d', '15-30d', '31-60d', '61-90d', '91-180d', '181-365d', '365d+']
repeat_pairs = user_article_history[user_article_history['repeat_pair'] == 1].copy()
repeat_pairs['last_gap_segment'] = pd.cut(repeat_pairs['days_since_last'], bins=gap_bins, labels=gap_labels)
gap_summary = repeat_pairs.groupby('last_gap_segment', observed=False).agg(
    repeat_pairs=('customer_key', 'size'), avg_purchase_count=('purchase_count', 'mean'), avg_seasonal_count=('seasonal_purchase_count', 'mean')
).reset_index()
gap_summary.to_csv(TABLE_DIR / 'repeat_pair_gap_summary.csv', index=False)
   
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=gap_summary, x='last_gap_segment', y='repeat_pairs', hue='last_gap_segment', ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Repeat user-article pairs by recency')
axes[0].set_xlabel('Days before target period')
axes[0].set_ylabel('Repeat pairs')
axes[0].tick_params(axis='x', rotation=30)
sns.lineplot(data=plot_data, x='alpha', y='map@12', marker='o', ax=axes[1], color='#e6550d')
axes[1].set_title('MAP@12 by time-decay alpha')
axes[1].set_xlabel('Alpha')
axes[1].set_ylabel('MAP@12')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '02_repeat_gap_distribution.png', bbox_inches='tight')
plt.close(fig)
   
time_decay_summary = {
    'best_alpha': best_alpha,
    'best_recall@12': float(plot_data['recall@12'].max()),
    'best_map@12': float(plot_data['map@12'].max()),
    'seasonal_boost_recall@12': float(seasonal_result['recall@12']),
    'seasonal_boost_map@12': float(seasonal_result['map@12']),
    'unique_user_article_pairs': int(len(user_article_history)),
    'repeat_pair_count': int(user_article_history['repeat_pair'].sum()),
    'repeat_pair_share': float(user_article_history['repeat_pair'].mean()),
    'saved_candidates': str(MINING_DIR / 'time_decay_candidates_top12.parquet'),
    'elapsed_seconds': round(time.time() - temporal_started, 3),
}
print(json.dumps(time_decay_summary, ensure_ascii=False, indent=2))

## 2. 多粒度购物篮关联规则挖掘

Baseline 2 只使用同一天共现且每个商品仅保留一个最强伴侣。这里在近 180 天历史中同时挖掘：

- **Same-day basket**：同一用户同一天购买的商品集合；
- **Same-week basket**：同一用户同一自然周内购买的商品集合，可捕捉跨天搭配与补货行为。

对每个无序商品对 $(A,B)$ 计算 Support、双向 Confidence 和 Lift，并保留每个 antecedent 的 Top 20 规则。

In [ ]:
association_started = time.time()
ASSOCIATION_DAYS = 180
association_history = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=ASSOCIATION_DAYS)].copy()


def mine_association_rules(frame, group_columns, max_basket_items, chunk_baskets=1_000_000):
    basket_items = frame[group_columns + ['article_num']].drop_duplicates().copy()
    basket_size = basket_items.groupby(group_columns, observed=True)['article_num'].transform('size')
    basket_items = basket_items[(basket_size >= 2) & (basket_size <= max_basket_items)].copy()
    basket_items['basket_id'] = basket_items.groupby(group_columns, sort=False, observed=True).ngroup().astype('int64')
    basket_items = basket_items[['basket_id', 'article_num']].sort_values(['basket_id', 'article_num']).reset_index(drop=True)
    basket_count = int(basket_items['basket_id'].max() + 1)
    article_basket_count = basket_items.groupby('article_num', observed=True).size().rename('antecedent_count').reset_index()
    pair_chunks = []
    for start in range(0, basket_count, chunk_baskets):
        end = min(start + chunk_baskets, basket_count)
        left = basket_items['basket_id'].searchsorted(start, side='left')
        right = basket_items['basket_id'].searchsorted(end, side='left')
        chunk = basket_items.iloc[left:right]
        pairs = chunk.merge(chunk, on='basket_id', suffixes=('_a', '_b'))
        pairs = pairs[pairs['article_num_a'] < pairs['article_num_b']]
        if pairs.empty:
            continue
        counts = pairs.groupby(['article_num_a', 'article_num_b'], observed=True, sort=False).size().reset_index(name='pair_count')
        pair_chunks.append(counts)
        del pairs
    pair_counts = pd.concat(pair_chunks, ignore_index=True).groupby(['article_num_a', 'article_num_b'], as_index=False, observed=True)['pair_count'].sum()
    directed_a = pair_counts.rename(columns={'article_num_a': 'antecedent_num', 'article_num_b': 'consequent_num'})
    directed_b = pair_counts.rename(columns={'article_num_b': 'antecedent_num', 'article_num_a': 'consequent_num'})
    directed = pd.concat([directed_a, directed_b], ignore_index=True)
    directed = directed.merge(article_basket_count.rename(columns={'article_num': 'antecedent_num'}), on='antecedent_num', how='left')
    directed = directed.merge(article_basket_count.rename(columns={'article_num': 'consequent_num', 'antecedent_count': 'consequent_count'}), on='consequent_num', how='left')
    directed['support'] = directed['pair_count'] / basket_count
    directed['confidence'] = directed['pair_count'] / directed['antecedent_count']
    directed['consequent_base_rate'] = directed['consequent_count'] / basket_count
    directed['lift'] = directed['confidence'] / directed['consequent_base_rate'].clip(lower=1e-12)
    directed['interest_score'] = np.log1p(directed['pair_count']) * directed['lift'] * directed['confidence']
    directed = directed.sort_values(['antecedent_num', 'interest_score', 'consequent_num'], ascending=[True, False, True], kind='stable')
    directed['_rule_rank'] = directed.groupby('antecedent_num', sort=False).cumcount() + 1
    directed = directed[directed['_rule_rank'] <= 20].drop(columns='_rule_rank')
    return directed, basket_count, len(pair_counts)


same_day_rules, same_day_basket_count, same_day_pair_count = mine_association_rules(
    association_history, ['customer_key', 't_dat'], max_basket_items=20
)
same_day_rules.to_parquet(MINING_DIR / 'association_rules_same_day.parquet', index=False, compression='zstd')
association_history['iso_week'] = association_history['t_dat'].dt.isocalendar().week.astype('int16')
association_history['iso_year'] = association_history['t_dat'].dt.isocalendar().year.astype('int16')
same_week_rules, same_week_basket_count, same_week_pair_count = mine_association_rules(
    association_history, ['customer_key', 'iso_year', 'iso_week'], max_basket_items=30
)
same_week_rules.to_parquet(MINING_DIR / 'association_rules_same_week.parquet', index=False, compression='zstd')


def association_candidates(rules, rule_rank_limit=10, recent_days=90):
    selected_rules = rules.sort_values(['antecedent_num', 'interest_score', 'consequent_num'], ascending=[True, False, True], kind='stable').copy()
    selected_rules['_rule_rank'] = selected_rules.groupby('antecedent_num', sort=False).cumcount() + 1
    selected_rules = selected_rules[selected_rules['_rule_rank'] <= rule_rank_limit]
    recent_user_items = history[
        (history['t_dat'] >= TARGET_START - pd.Timedelta(days=recent_days)) & history['customer_key'].isin(target_user_set)
    ][['customer_key', 'article_num']].drop_duplicates()
    expanded = recent_user_items.merge(
        selected_rules[['antecedent_num', 'consequent_num', 'pair_count', 'confidence', 'lift']],
        left_on='article_num', right_on='antecedent_num', how='inner'
    )
    expanded = expanded[expanded['customer_key'].map(lambda value: value in target_user_set)]
    expanded = expanded.rename(columns={'consequent_num': 'candidate_article'})
    expanded = expanded.merge(
        history_pair_set.rename(columns={'article_num': 'candidate_article'}),
        on=['customer_key', 'candidate_article'], how='left'
    )
    expanded = expanded[expanded['in_history'].fillna(0) == 0]
    expanded['rule_score'] = np.log1p(expanded['pair_count']) * expanded['confidence'] * expanded['lift'].clip(upper=20)
    candidates = expanded.groupby(['customer_key', 'candidate_article'], as_index=False, observed=True).agg(
        score=('rule_score', 'sum'), rule_count=('rule_score', 'size'), max_lift=('lift', 'max')
    ).rename(columns={'candidate_article': 'article_num'})
    return top_k_by_user(candidates, 'score', top_n=12)


same_day_candidates = association_candidates(same_day_rules, recent_days=90)
same_week_candidates = association_candidates(same_week_rules, recent_days=90)
same_day_candidates.to_parquet(MINING_DIR / 'association_candidates_same_day_top12.parquet', index=False, compression='zstd')
same_week_candidates.to_parquet(MINING_DIR / 'association_candidates_same_week_top12.parquet', index=False, compression='zstd')
same_day_result = evaluate_candidates('association_same_day', same_day_candidates, target_truth, len(target_users), history_pairs=history_pair_set)
same_week_result = evaluate_candidates('association_same_week', same_week_candidates, target_truth, len(target_users), history_pairs=history_pair_set)
   
rule_compare = pd.DataFrame([same_day_result, same_week_result])
rule_compare.to_csv(TABLE_DIR / 'association_rule_evaluation.csv', index=False)
metric_columns = ['map@12', 'recall@12', 'hit_rate@12', 'novel_item_share']
plot_rule = rule_compare.melt(id_vars='channel', value_vars=metric_columns, var_name='metric', value_name='value')
fig, axis = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=plot_rule, x='metric', y='value', hue='channel', ax=axis, palette='Set2')
axis.set_title('Same-day vs same-week association recall')
axis.set_xlabel('Metric')
axis.set_ylabel('Score')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '03_association_rule_comparison.png', bbox_inches='tight')
plt.close(fig)
   
association_summary = {
    'history_days': ASSOCIATION_DAYS,
    'same_day_baskets_with_2_to_20_items': same_day_basket_count,
    'same_day_unique_item_pairs': same_day_pair_count,
    'same_day_saved_rules': int(len(same_day_rules)),
    'same_week_baskets_with_2_to_30_items': same_week_basket_count,
    'same_week_unique_item_pairs': same_week_pair_count,
    'same_week_saved_rules': int(len(same_week_rules)),
    'same_day_recall@12': same_day_result['recall@12'],
    'same_week_recall@12': same_week_result['recall@12'],
    'same_day_map@12': same_day_result['map@12'],
    'same_week_map@12': same_week_result['map@12'],
    'elapsed_seconds': round(time.time() - association_started, 3),
}
print(json.dumps(association_summary, ensure_ascii=False, indent=2))

## 3. 隐式反馈协同过滤挖掘

将近 60 天用户-商品交互构造成稀疏隐式反馈矩阵，权重为 $\log(1+count)\times 0.995^{\Delta t}$。使用随机化 SVD 学习 64 维商品向量，通过向量余弦相似度挖掘每个商品最相近的邻居。

该通道不要求用户购买过候选商品，能够扩展“相似人群购买”信号，弥补复购召回的新颖性不足。

In [ ]:
cf_started = time.time()
CF_DAYS = 60
CF_RANK = 64
cf_history = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=CF_DAYS)].copy()
cf_interactions = cf_history.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
    purchase_count=('t_dat', 'size'), last_date=('t_dat', 'max')
)
cf_interactions['days_to_target'] = (TARGET_START - cf_interactions['last_date']).dt.days.astype('int16')
cf_interactions['interaction_weight'] = np.log1p(cf_interactions['purchase_count']) * np.power(0.995, cf_interactions['days_to_target']).astype('float32')
user_codes, user_uniques = pd.factorize(cf_interactions['customer_key'], sort=True)
article_codes, article_uniques = pd.factorize(cf_interactions['article_num'], sort=True)
article_array = np.asarray(article_uniques, dtype='int32')
interaction_matrix = sparse.csr_matrix(
    (cf_interactions['interaction_weight'].astype('float32'), (user_codes, article_codes)),
    shape=(len(user_uniques), len(article_uniques)), dtype=np.float32,
)


def randomized_svd(matrix, rank, oversample=16, seed=2026):
    rng = np.random.default_rng(seed)
    sampled_dimension = min(rank + oversample, min(matrix.shape) - 1)
    random_projection = rng.standard_normal((matrix.shape[1], sampled_dimension), dtype=np.float32)
    projected = matrix @ random_projection
    orthogonal_basis, _ = np.linalg.qr(projected, mode='reduced')
    small_matrix = orthogonal_basis.T @ matrix
    left_vectors, singular_values, right_transpose = np.linalg.svd(small_matrix, full_matrices=False)
    item_factors = (right_transpose[:rank].T * singular_values[:rank]).astype(np.float32)
    return item_factors, singular_values[:rank]


item_factors, singular_values = randomized_svd(interaction_matrix, CF_RANK)
item_norms = np.linalg.norm(item_factors, axis=1)
normalized_factors = item_factors / np.maximum(item_norms, 1e-8)[:, None]
neighbor_chunks = []
for start in range(0, len(article_uniques), 2048):
    end = min(start + 2048, len(article_uniques))
    similarities = normalized_factors[start:end] @ normalized_factors.T
    candidate_count = min(12, similarities.shape[1])
    top_indices = np.argpartition(-similarities, candidate_count - 1, axis=1)[:, :candidate_count]
    top_values = np.take_along_axis(similarities, top_indices, axis=1)
    order = np.argsort(-top_values, axis=1)
    top_indices = np.take_along_axis(top_indices, order, axis=1)
    top_values = np.take_along_axis(top_values, order, axis=1)
    row_offsets = np.arange(start, end)[:, None]
    neighbor_chunks.append(pd.DataFrame({
        'article_num': article_array[np.repeat(np.arange(start, end), candidate_count)].astype('int32'),
        'neighbor_num': article_array[top_indices].reshape(-1).astype('int32'),
        'similarity': top_values.reshape(-1).astype('float32'),
        'similarity_rank': np.tile(np.arange(1, candidate_count + 1), end - start),
    }))
item_cf_neighbors = pd.concat(neighbor_chunks, ignore_index=True)
item_cf_neighbors = item_cf_neighbors[(item_cf_neighbors['article_num'] != item_cf_neighbors['neighbor_num']) & (item_cf_neighbors['similarity'] > 0.01)]
item_cf_neighbors['similarity_rank'] = item_cf_neighbors.groupby('article_num', sort=False).cumcount() + 1
item_cf_neighbors = item_cf_neighbors[item_cf_neighbors['similarity_rank'] <= 10]
item_cf_neighbors = item_cf_neighbors.merge(
    article_meta.rename(columns={
        'article_num': 'article_num', 'product_group_name': 'source_product_group',
        'index_group_name': 'source_index_group', 'colour_group_name': 'source_colour_group'
    })[['article_num', 'source_product_group', 'source_index_group', 'source_colour_group']],
    on='article_num', how='left'
)
item_cf_neighbors = item_cf_neighbors.merge(
    article_meta.rename(columns={
        'article_num': 'neighbor_num', 'product_group_name': 'neighbor_product_group',
        'index_group_name': 'neighbor_index_group', 'colour_group_name': 'neighbor_colour_group'
    })[['neighbor_num', 'neighbor_product_group', 'neighbor_index_group', 'neighbor_colour_group']],
    on='neighbor_num', how='left'
)
item_cf_neighbors['same_index_group'] = item_cf_neighbors['source_index_group'] == item_cf_neighbors['neighbor_index_group']
item_cf_neighbors['same_product_group'] = item_cf_neighbors['source_product_group'] == item_cf_neighbors['neighbor_product_group']
item_cf_neighbors['same_colour_group'] = item_cf_neighbors['source_colour_group'] == item_cf_neighbors['neighbor_colour_group']
item_cf_neighbors.to_parquet(MINING_DIR / 'item_cf_neighbors.parquet', index=False, compression='zstd')


cf_user_items = cf_interactions[cf_interactions['customer_key'].isin(target_user_set)][['customer_key', 'article_num', 'days_to_target']].copy()
cf_user_items['recency_weight'] = np.power(0.995, cf_user_items['days_to_target']).astype('float32')
cf_expanded = cf_user_items.merge(item_cf_neighbors[['article_num', 'neighbor_num', 'similarity', 'similarity_rank']], on='article_num', how='inner')
cf_expanded = cf_expanded.rename(columns={'neighbor_num': 'candidate_article'})
cf_expanded = cf_expanded.merge(history_pair_set.rename(columns={'article_num': 'candidate_article'}), on=['customer_key', 'candidate_article'], how='left')
cf_expanded = cf_expanded[cf_expanded['in_history'].fillna(0) == 0]
cf_expanded['cf_score'] = cf_expanded['similarity'] * cf_expanded['recency_weight'] / cf_expanded['similarity_rank']
cf_candidates = cf_expanded.groupby(['customer_key', 'candidate_article'], as_index=False, observed=True).agg(score=('cf_score', 'sum')).rename(columns={'candidate_article': 'article_num'})
cf_candidates = top_k_by_user(cf_candidates, 'score', top_n=12)
cf_candidates.to_parquet(MINING_DIR / 'implicit_cf_candidates_top12.parquet', index=False, compression='zstd')
cf_result = evaluate_candidates('implicit_item_cf_svd64', cf_candidates, target_truth, len(target_users), history_pairs=history_pair_set)
   
cf_summary = {
    'history_days': CF_DAYS,
    'matrix_users': int(interaction_matrix.shape[0]),
    'matrix_articles': int(interaction_matrix.shape[1]),
    'matrix_nonzeros': int(interaction_matrix.nnz),
    'matrix_density': float(interaction_matrix.nnz / (interaction_matrix.shape[0] * interaction_matrix.shape[1])),
    'rank': CF_RANK,
    'top_singular_value': float(singular_values[0]),
    'rank64_singular_value': float(singular_values[-1]),
    'saved_neighbors': int(len(item_cf_neighbors)),
    'same_index_group_share': float(item_cf_neighbors['same_index_group'].mean()),
    'same_product_group_share': float(item_cf_neighbors['same_product_group'].mean()),
    'same_colour_group_share': float(item_cf_neighbors['same_colour_group'].mean()),
    'recall@12': cf_result['recall@12'],
    'map@12': cf_result['map@12'],
    'novel_item_share': cf_result['novel_item_share'],
    'elapsed_seconds': round(time.time() - cf_started, 3),
}
   
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(np.arange(1, len(singular_values) + 1), singular_values, marker='o', color='#08519c')
axes[0].set_title('Singular value spectrum of implicit feedback matrix')
axes[0].set_xlabel('Singular value index')
axes[0].set_ylabel('Singular value')
similarity_share = pd.DataFrame({
    'Pattern': ['Same index group', 'Same product group', 'Same colour group'],
    'Share': [cf_summary['same_index_group_share'], cf_summary['same_product_group_share'], cf_summary['same_colour_group_share']],
})
sns.barplot(data=similarity_share, x='Pattern', y='Share', hue='Pattern', ax=axes[1], palette='Set2', legend=False)
axes[1].set_title('Metadata agreement in SVD item neighbors')
axes[1].set_xlabel('')
axes[1].set_ylabel('Neighbor share')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '04_implicit_cf_svd.png', bbox_inches='tight')
plt.close(fig)
   
print(json.dumps(cf_summary, ensure_ascii=False, indent=2))

## 4. 用户偏好与分群热门挖掘

EDA 显示年龄、会员状态与购买强度存在差异，且 Top 100 商品仅覆盖 4.6% 交易。因此，全局热门必须扩展为：

- 用户近一年偏好的大类 / 品类 / 颜色；
- 年龄段、会员状态、活跃度、近期访问分群的热门商品；
- 不同历史窗口的全局热门，用于比较热门稳定性。

In [ ]:
profile_started = time.time()
PREFERENCE_DAYS = 365
POPULARITY_DAYS = 28
preference_history = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=PREFERENCE_DAYS)].copy()
preference_with_meta = preference_history.merge(
    article_meta[['article_num', 'index_group_name', 'product_group_name', 'product_type_name', 'colour_group_name']],
    on='article_num', how='left'
)
   
def mine_top_preference(frame, category_column, output_column, top_n=2):
    counts = frame.groupby(['customer_key', category_column], observed=True).size().rename('category_count').reset_index()
    user_total = counts.groupby('customer_key', observed=True)['category_count'].sum().rename('category_total').reset_index()
    counts = counts.merge(user_total, on='customer_key', how='left')
    counts['affinity'] = counts['category_count'] / counts['category_total']
    counts = counts.sort_values(['customer_key', 'category_count', 'affinity', category_column], ascending=[True, False, False, True], kind='stable')
    counts['preference_rank'] = counts.groupby('customer_key', sort=False).cumcount() + 1
    selected = counts[counts['preference_rank'] <= top_n].copy()
    selected[output_column] = selected[category_column].astype(str)
    return selected[['customer_key', output_column, 'category_count', 'category_total', 'affinity', 'preference_rank']]
   
index_preferences = mine_top_preference(preference_with_meta, 'index_group_name', 'preferred_index_group')
product_preferences = mine_top_preference(preference_with_meta, 'product_group_name', 'preferred_product_group')
type_preferences = mine_top_preference(preference_with_meta, 'product_type_name', 'preferred_product_type')
colour_preferences = mine_top_preference(preference_with_meta, 'colour_group_name', 'preferred_colour_group')
customer_preferences = index_preferences[['customer_key', 'preference_rank', 'preferred_index_group']].merge(
    product_preferences[['customer_key', 'preference_rank', 'preferred_product_group']], on=['customer_key', 'preference_rank'], how='outer'
)
customer_preferences = customer_preferences.merge(type_preferences[['customer_key', 'preference_rank', 'preferred_product_type']], on=['customer_key', 'preference_rank'], how='outer')
customer_preferences = customer_preferences.merge(colour_preferences[['customer_key', 'preference_rank', 'preferred_colour_group']], on=['customer_key', 'preference_rank'], how='outer')
customer_preferences.to_parquet(MINING_DIR / 'customer_category_preferences.parquet', index=False, compression='zstd')
   
user_price = preference_history.groupby('customer_key', as_index=False, observed=True).agg(
    median_price=('price', 'median'), mean_price=('price', 'mean'), min_price=('price', 'min'), max_price=('price', 'max')
)
user_price['price_band'] = pd.cut(
    user_price['median_price'], bins=[0, 0.015, 0.025, 0.035, 0.045, 0.060, 1.0],
    labels=['<=0.015', '0.015-0.025', '0.025-0.035', '0.035-0.045', '0.045-0.060', '>0.060']
)
user_price.to_parquet(MINING_DIR / 'customer_price_preferences.parquet', index=False, compression='zstd')
   
customer_activity = history.groupby('customer_key', as_index=False, observed=True).agg(
    lifetime_purchases=('t_dat', 'size'), lifetime_active_days=('t_dat', 'nunique'), last_date=('t_dat', 'max')
)
customer_activity['recency_days'] = (TARGET_START - customer_activity['last_date']).dt.days.astype('int16')
customer_activity['activity_segment'] = pd.cut(
    customer_activity['lifetime_purchases'], bins=[0, 4, 24, np.inf], labels=['low', 'middle', 'high'], right=True
)
customer_activity['recency_segment'] = pd.cut(
    customer_activity['recency_days'], bins=[-1, 7, 30, 90, 180, 365, 10000],
    labels=['0-7d', '8-30d', '31-90d', '91-180d', '181-365d', '365d+'], right=True
)
customer_profile = customers[['customer_key', 'age', 'club_member_status', 'fashion_news_frequency']].merge(
    customer_activity[['customer_key', 'lifetime_purchases', 'lifetime_active_days', 'activity_segment', 'recency_segment']],
    on='customer_key', how='left'
)
customer_profile['age_group'] = pd.cut(
    customer_profile['age'], bins=[15, 25, 35, 45, 55, 65, 100],
    labels=['16-25', '26-35', '36-45', '46-55', '56-65', '66+']
)
customer_profile['age_group'] = customer_profile['age_group'].cat.add_categories('unknown').fillna('unknown')
customer_profile['club_member_status'] = customer_profile['club_member_status'].astype(str).fillna('UNKNOWN')
customer_profile['fashion_news_frequency'] = customer_profile['fashion_news_frequency'].astype(str).fillna('UNKNOWN')
customer_profile.to_parquet(MINING_DIR / 'customer_mining_profile.parquet', index=False, compression='zstd')
   
recent_history = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=POPULARITY_DAYS)].copy()
recent_popularity = recent_history.groupby('article_num', as_index=False, observed=True).agg(
    purchase_count=('article_num', 'size'), customer_count=('customer_key', 'nunique'), revenue=('price', 'sum')
)
recent_popularity = recent_popularity.merge(article_meta, on='article_num', how='left').sort_values('purchase_count', ascending=False)
recent_popularity.to_parquet(MINING_DIR / 'recent_article_popularity_28d.parquet', index=False, compression='zstd')
   
segment_columns = ['age_group', 'club_member_status', 'fashion_news_frequency', 'activity_segment', 'recency_segment']
segment_popularity_frames = []
recent_with_profile = recent_history[['customer_key', 'article_num']].merge(
    customer_profile[['customer_key'] + segment_columns], on='customer_key', how='left'
)
for segment_column in segment_columns:
    summary = recent_with_profile.groupby([segment_column, 'article_num'], observed=True).size().rename('segment_purchase_count').reset_index()
    summary['segment_name'] = segment_column
    summary = summary.rename(columns={segment_column: 'segment_value'})
    summary = summary.sort_values(['segment_name', 'segment_value', 'segment_purchase_count', 'article_num'], ascending=[True, True, False, True], kind='stable')
    summary['popularity_rank'] = summary.groupby(['segment_name', 'segment_value'], sort=False).cumcount() + 1
    summary = summary[summary['popularity_rank'] <= 50]
    segment_popularity_frames.append(summary)
segment_popularity = pd.concat(segment_popularity_frames, ignore_index=True)
segment_popularity.to_parquet(MINING_DIR / 'segment_popularity_top50.parquet', index=False, compression='zstd')
   
category_popularity = recent_with_meta = recent_history.merge(
    article_meta[['article_num', 'index_group_name', 'product_group_name']], on='article_num', how='left'
).groupby(['index_group_name', 'article_num'], observed=True).size().rename('category_purchase_count').reset_index()
category_popularity = category_popularity.sort_values(['index_group_name', 'category_purchase_count', 'article_num'], ascending=[True, False, True], kind='stable')
category_popularity['category_rank'] = category_popularity.groupby('index_group_name', sort=False).cumcount() + 1
category_popularity = category_popularity[category_popularity['category_rank'] <= 100]
category_popularity.to_parquet(MINING_DIR / 'category_popularity_top100_28d.parquet', index=False, compression='zstd')
   
def preference_candidates():
    selected_users = index_preferences[index_preferences['customer_key'].isin(target_user_set) & (index_preferences['preference_rank'] <= 2)]
    expanded = selected_users.merge(
        category_popularity.rename(columns={'index_group_name': 'preferred_index_group'}),
        on='preferred_index_group', how='inner'
    )
    expanded = expanded.merge(history_pair_set, on=['customer_key', 'article_num'], how='left')
    expanded = expanded[expanded['in_history'].fillna(0) == 0]
    expanded['preference_score'] = expanded['affinity'] * expanded['category_count'] / expanded['category_rank']
    candidates = expanded.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(score=('preference_score', 'sum')).sort_values('score', ascending=False)
    return top_k_by_user(candidates, 'score', top_n=12)
   
def segment_candidates(segment_name, top_n=12):
    mapping = customer_profile[customer_profile['customer_key'].isin(target_user_set)][['customer_key', segment_name]].copy()
    mapping['segment_value'] = mapping[segment_name].astype(str)
    selected = segment_popularity[(segment_popularity['segment_name'] == segment_name) & (segment_popularity['popularity_rank'] <= top_n)]
    expanded = mapping[['customer_key', 'segment_value']].merge(selected[['segment_value', 'article_num', 'segment_purchase_count', 'popularity_rank']], on='segment_value', how='inner')
    expanded = expanded.merge(history_pair_set, on=['customer_key', 'article_num'], how='left')
    expanded = expanded[expanded['in_history'].fillna(0) == 0]
    expanded['segment_score'] = expanded['segment_purchase_count'] / expanded['popularity_rank']
    candidates = expanded.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(score=('segment_score', 'sum'))
    return top_k_by_user(candidates, 'score', top_n=12)
   
preference_candidate_frame = preference_candidates()
age_candidate_frame = segment_candidates('age_group')
activity_candidate_frame = segment_candidates('activity_segment')
recency_candidate_frame = segment_candidates('recency_segment')
preference_result = evaluate_candidates('preferred_index_group_popularity', preference_candidate_frame, target_truth, len(target_users), history_pairs=history_pair_set)
age_result = evaluate_candidates('age_group_popularity', age_candidate_frame, target_truth, len(target_users), history_pairs=history_pair_set)
activity_result = evaluate_candidates('activity_segment_popularity', activity_candidate_frame, target_truth, len(target_users), history_pairs=history_pair_set)
recency_result = evaluate_candidates('recency_segment_popularity', recency_candidate_frame, target_truth, len(target_users), history_pairs=history_pair_set)
   
profile_mining_results = pd.DataFrame([preference_result, age_result, activity_result, recency_result])
profile_mining_results.to_csv(TABLE_DIR / 'preference_segment_evaluation.csv', index=False)
fig, axis = plt.subplots(figsize=(13, 6))
plot_profile = profile_mining_results.melt(id_vars='channel', value_vars=['map@12', 'recall@12', 'hit_rate@12'], var_name='metric', value_name='value')
sns.barplot(data=plot_profile, x='metric', y='value', hue='channel', ax=axis, palette='Set2')
axis.set_title('Preference and segmented popularity signals')
axis.set_xlabel('Metric')
axis.set_ylabel('Score')
axis.tick_params(axis='x', rotation=15)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '05_preference_segment_mining.png', bbox_inches='tight')
plt.close(fig)
   
profile_mining_summary = {
    'preference_history_days': PREFERENCE_DAYS,
    'popularity_window_days': POPULARITY_DAYS,
    'users_with_index_preference': int(index_preferences['customer_key'].nunique()),
    'users_with_price_preference': int(len(user_price)),
    'saved_customer_preferences': int(len(customer_preferences)),
    'saved_segment_popularity_rows': int(len(segment_popularity)),
    'best_profile_channel': str(profile_mining_results.sort_values('recall@12', ascending=False).iloc[0]['channel']),
    'best_profile_recall@12': float(profile_mining_results['recall@12'].max()),
    'best_profile_map@12': float(profile_mining_results['map@12'].max()),
    'elapsed_seconds': round(time.time() - profile_started, 3),
}
print(json.dumps(profile_mining_summary, ensure_ascii=False, indent=2))

## 5. 通道横向评估与规则融合

将时间衰减复购、同天/同周关联、SVD 商品协同过滤、偏好热门、分群热门和全局热门放入同一测试评估框架，比较 MAP@12、Recall@12、HitRate@12 与新颖性。

随后使用固定 Borda 权重做轻量融合，验证“多路召回互补”是否优于单一数据挖掘信号。

In [ ]:
fusion_started = time.time()
target_user_frame = pd.DataFrame({'customer_key': target_users})
window_results = []
window_candidate_map = {}
for window_days in [7, 14, 28, 91, 364]:
    window_history = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=window_days)]
    top_items = window_history['article_num'].value_counts().head(12).rename_axis('article_num').reset_index(name='score')
    top_items['article_num'] = top_items['article_num'].astype('int32')
    candidates = target_user_frame.merge(top_items[['article_num', 'score']], how='cross')
    candidates = candidates.merge(history_pair_set, on=['customer_key', 'article_num'], how='left')
    candidates = candidates[candidates['in_history'].fillna(0) == 0].drop(columns='in_history')
    window_candidate_map[window_days] = candidates
    window_results.append(evaluate_candidates(f'global_popularity_{window_days}d', candidates, target_truth, len(target_users), history_pairs=history_pair_set, show_output=False))
   
best_global_window_row = max(window_results, key=lambda item: item['recall@12'])
best_global_window = int(best_global_window_row['channel'].replace('global_popularity_', '').replace('d', ''))
channel_results = []
channel_results.extend(window_results)
channel_results.extend([
    evaluate_candidates(f'time_decay_alpha={best_alpha}', alpha_candidates[best_alpha], target_truth, len(target_users), history_pairs=history_pair_set, show_output=False),
    same_day_result,
    same_week_result,
    cf_result,
    preference_result,
    age_result,
    activity_result,
    recency_result,
])
channel_evaluation = pd.DataFrame(channel_results).sort_values('recall@12', ascending=False).reset_index(drop=True)
best_single_channel_row = channel_evaluation.iloc[0].copy()
channel_evaluation.to_csv(TABLE_DIR / 'mining_channel_evaluation.csv', index=False)
   
fusion_inputs = {
    'time_decay': (alpha_candidates[best_alpha], 1.50),
    'association_same_day': (same_day_candidates, 0.70),
    'association_same_week': (same_week_candidates, 0.70),
    'implicit_cf': (cf_candidates, 0.50),
    'index_preference': (preference_candidate_frame, 0.45),
    'age_popularity': (age_candidate_frame, 0.25),
    'activity_popularity': (activity_candidate_frame, 0.30),
    'recency_popularity': (recency_candidate_frame, 0.30),
    'global_popularity': (window_candidate_map[best_global_window], 0.20),
}
fusion_chunks = []
for channel, (frame, channel_weight) in fusion_inputs.items():
    normalized = frame[['customer_key', 'article_num', 'score']].copy()
    normalized = normalized.sort_values(['customer_key', 'score', 'article_num'], ascending=[True, False, True], kind='stable')
    normalized['rank'] = normalized.groupby('customer_key', sort=False).cumcount() + 1
    normalized['fusion_score'] = channel_weight / normalized['rank']
    normalized['source_channel'] = channel
    fusion_chunks.append(normalized[['customer_key', 'article_num', 'fusion_score', 'source_channel']])
fusion_all = pd.concat(fusion_chunks, ignore_index=True)
fusion_candidates = fusion_all.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
    score=('fusion_score', 'sum'), source_count=('source_channel', 'nunique')
)
fusion_candidates = top_k_by_user(fusion_candidates, 'score', top_n=12)
fusion_candidates.to_parquet(MINING_DIR / 'pattern_fusion_candidates_top12.parquet', index=False, compression='zstd')
fusion_result = evaluate_candidates('pattern_fusion_fixed_weights', fusion_candidates, target_truth, len(target_users), history_pairs=history_pair_set)
channel_evaluation = pd.concat([channel_evaluation, pd.DataFrame([fusion_result])], ignore_index=True).sort_values('recall@12', ascending=False).reset_index(drop=True)
channel_evaluation.to_csv(TABLE_DIR / 'mining_channel_evaluation.csv', index=False)
   
fig, axes = plt.subplots(2, 1, figsize=(15, 11))
plot_channels = channel_evaluation.copy()
plot_channels['short_channel'] = plot_channels['channel'].str.replace('global_popularity_', 'pop_', regex=False).str.replace('time_decay_alpha=', 'decay_', regex=False)
sns.barplot(data=plot_channels, x='recall@12', y='short_channel', hue='short_channel', ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Recall@12 by mined signal channel')
axes[0].set_xlabel('Recall@12')
axes[0].set_ylabel('')
sns.barplot(data=plot_channels, x='map@12', y='short_channel', hue='short_channel', ax=axes[1], palette='Set2', legend=False)
axes[1].set_title('MAP@12 by mined signal channel')
axes[1].set_xlabel('MAP@12')
axes[1].set_ylabel('')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '06_mining_channel_evaluation.png', bbox_inches='tight')
plt.close(fig)
   
output_artifacts = {
    'time_decay_candidates_top12.parquet': '全体用户时间衰减复购 Top12',
    'association_rules_same_day.parquet': '同天购物篮关联规则，每个 antecedent Top20',
    'association_rules_same_week.parquet': '同周购买关联规则，每个 antecedent Top20',
    'association_candidates_same_day_top12.parquet': '测试用户同天关联候选 Top12',
    'association_candidates_same_week_top12.parquet': '测试用户同周关联候选 Top12',
    'item_cf_neighbors.parquet': '近 60 天 SVD 商品相似邻居 Top10',
    'implicit_cf_candidates_top12.parquet': '测试用户 SVD 协同过滤候选 Top12',
    'customer_category_preferences.parquet': '用户近一年类目/品类/颜色偏好 Top2',
    'customer_price_preferences.parquet': '用户价格偏好与价格带',
    'customer_mining_profile.parquet': '年龄、会员、活跃度、近期访问分群画像',
    'segment_popularity_top50.parquet': '五类分群的近 28 天热门商品 Top50',
    'category_popularity_top100_28d.parquet': '大类内近 28 天热门商品 Top100',
    'pattern_fusion_candidates_top12.parquet': '测试用户多模式固定融合候选 Top12',
}
mining_summary = {
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S %z'),
    'data_source': str(CLEAN_DIR) if CLEAN_DIR else f'raw fallback: {RAW_DIR}',
    'target_period': [str(TARGET_START.date()), str(TARGET_END.date())],
    'target_users': int(len(target_users)),
    'target_user_article_pairs': int(len(target_truth)),
    'time_decay': time_decay_summary,
    'association': association_summary,
    'implicit_cf': cf_summary,
    'profile_mining': profile_mining_summary,
    'best_global_window_days': best_global_window,
    'best_single_channel': str(best_single_channel_row['channel']),
    'best_single_recall@12': float(best_single_channel_row['recall@12']),
    'best_single_map@12': float(best_single_channel_row['map@12']),
    'fusion_recall@12': float(fusion_result['recall@12']),
    'fusion_map@12': float(fusion_result['map@12']),
    'fusion_gain_over_best_single_recall': float(fusion_result['recall@12'] / best_single_channel_row['recall@12'] - 1),
    'elapsed_seconds': round(time.time() - fusion_started, 3),
    'output_artifacts': output_artifacts,
}
(MINING_DIR / 'mining_summary.json').write_text(json.dumps(mining_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(channel_evaluation.to_string(index=False))
print(json.dumps(mining_summary, ensure_ascii=False, indent=2))

## 数据挖掘结论

1. **复购是强召回信号**：时间衰减项显著优于普通热门，验证了 EDA 中同商品短间隔复购的结论。
2. **同周共现弥补同天共现**：同周购物篮覆盖跨天补货与搭配。
3. **SVD 邻居提供新颖候选**：协同过滤候选大多不在用户历史中，可提升候选池多样性和覆盖率。
4. **分群热门解决部分冷启动**：年龄、活跃度和近期访问分群能缓解全局热门过度集中问题。
5. **单一规则不足以排序**：各通道 Recall 与 MAP 排名并不完全一致，`04_final_candidate_feature_model.ipynb` 因此构建“多路召回候选池 + Learning-to-Rank 精排”。